# 📊 Trabalho Final: Redes Neurais Profundas — **Etapa 2**
**Universidade Federal de Goiás (UFG) — Instituto de Informática (INF)**

**Projeto:** Análise Arquitetural do Congelamento de Camadas na Mitigação de *Domain Shift* e *Language Shift* em Transformers Multilíngues

**Equipe:**
- Geovana Teixeira Camargo — 202303338
- Sebastião Corrêa Fraga Neto — 202301487
- Pedro Reis Pimenta — 202303359
- Henrique Matheus Mendonça de Miranda — 202402479

---
> **Status:** **Etapa 2 — Implementação do Modelo e Algoritmo de Congelamento.** Este notebook importa `src/` e lê os 5 CSVs da Etapa 1 **a partir do Google Drive** (o `/content` do Colab é efêmero). Implementa as Fases 2.0–2.4 do `PLAN-Etapa2.md`. **Conteúdo atual: Fases 2.0 a 2.4 — Etapa 2 completa.**


---
## 🟩 Etapa 2 — Modelo XLM-RoBERTa e Congelamento C1–C4

| Fase | Descrição | Status |
|------|-----------|--------|
| **2.0** | Setup do ambiente + carga dos 5 CSVs do Drive | ✅ |
| 2.1 | Arquitetura do classificador (`carregar_modelo`) → `src/model.py` | ✅ |
| 2.2 | Lógica de congelamento `freeze_layers` (C1–C4) | ✅ |
| 2.3 | Testes de verificação (`tests/test_model.py`) + tabela real de parâmetros | ✅ |
| 2.4 | Smoke test com dados reais (forward + backward; gradiente só nos treináveis) | ✅ |

**Decisões travadas:** notebook novo (D1) · *classification head* padrão do HF — `XLMRobertaForSequenceClassification` (D2) · seeds oficiais `{42, 123, 2024}`.


### 2.0 — Setup e carga dos dados

- **INPUT:** os 5 CSVs da Etapa 1, persistidos no Google Drive (`MyDrive/TrabalhoRNP/data_processed/`).
- **AÇÕES:** instalar libs · fixar seeds · verificar GPU · montar Drive · carregar os 5 DataFrames.
- **OUTPUT:** ambiente pronto (libs, GPU, seeds), Drive montado e os 5 DataFrames em memória (`dados`).
- **VERIFY:** tamanhos batem (treino ≈ 10,7k; cada célula de teste ≈ 2,68k), colunas corretas e classes balanceadas 50/50.

> ⚠️ **Pré-requisito:** a Etapa 1 deve ter copiado os CSVs para o Drive. Se ainda não fez, rode no fim do notebook da Etapa 1 a célula que monta o Drive e copia `/content/data_processed`, `/content/src` e `/content/b2w.csv` para `MyDrive/TrabalhoRNP/`.


In [ ]:
# 2.0 (1/5) — Bibliotecas (XLM-R, métricas, utilitários)
!pip install -q transformers accelerate evaluate scikit-learn matplotlib seaborn
print("Bibliotecas instaladas.")


In [ ]:
# 2.0 (2/5) — Hardware, seeds e verificação de GPU (mesmo padrão da Etapa 1)
import torch, random, os
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import set_seed
from IPython.display import display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware em uso: {device}")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"| memória: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️ Sem GPU. Em Ambiente de execução > Alterar o tipo de ambiente > Acelerador = GPU.")

SEEDS = [42, 123, 2024]   # seeds oficiais do projeto (usadas nas Etapas 3-4)

def fixar_seed(seed: int):
    """Fixa todas as fontes de aleatoriedade (idêntico à Etapa 1)."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    set_seed(seed)

fixar_seed(SEEDS[0])
print(f"Seeds do projeto: {SEEDS} | seed inicial fixada: {SEEDS[0]}")


In [ ]:
# 2.0 (3/5) — Montar o Drive e localizar o projeto (CSVs + src/ da Etapa 1)
from google.colab import drive
import sys

drive.mount('/content/drive')

# Raiz do projeto no Drive — ajuste se você usou outra pasta ao persistir na Etapa 1.
DIR_PROJ = Path('/content/drive/MyDrive/TrabalhoRNP')
DIR_DATA = DIR_PROJ / 'data_processed'

assert DIR_PROJ.exists(), (
    f"Pasta do projeto não encontrada: {DIR_PROJ}\n"
    "Rode a célula de persistência no FIM da Etapa 1 (monta o Drive e copia "
    "/content/data_processed, /content/src e /content/b2w.csv para o Drive).")
assert DIR_DATA.exists(), f"Sem data_processed em {DIR_PROJ}. Confira a persistência da Etapa 1."

# Deixa o src/ da Etapa 1 importável (data_pipeline agora; model.py virá na Fase 2.1).
if str(DIR_PROJ) not in sys.path:
    sys.path.insert(0, str(DIR_PROJ))

print("Projeto :", DIR_PROJ)
print("Dados   :", DIR_DATA)
print("CSVs    :", [p.name for p in sorted(DIR_DATA.glob('*.csv'))])


In [ ]:
# 2.0 (4/5) — Carregar os 5 CSVs em DataFrames
ARQUIVOS = {
    "S1_train": "S1_train_en_eletronicos.csv",   # treino (EN/Eletrônicos)
    "S1_val":   "S1_val_en_eletronicos.csv",     # T1 — controle interno
    "S2":       "S2_en_beleza.csv",              # T2 — Domain Shift
    "S3":       "S3_pt_eletronicos.csv",         # T3 — Language Shift
    "S4":       "S4_pt_beleza.csv",              # T4 — combinado
}
dados = {}
for nome, arq in ARQUIVOS.items():
    caminho = DIR_DATA / arq
    assert caminho.exists(), f"CSV ausente: {caminho}"
    dados[nome] = pd.read_csv(caminho)
    print(f"{nome:>9}: {len(dados[nome]):>7,} linhas | colunas = {list(dados[nome].columns)}")

print("\nAmostra de S1_train:")
display(dados["S1_train"].head(3))


In [ ]:
# 2.0 (5/5) — VERIFY: tamanhos, colunas e balanceamento de classes
COLS_ESPERADAS = {"id", "idioma", "dominio", "label", "texto"}

print(f"{'subset':>9} | {'linhas':>7} | {'neg':>6} | {'pos':>6} | balanceado")
print("-" * 52)
for nome, df in dados.items():
    faltando = COLS_ESPERADAS - set(df.columns)
    assert not faltando, f"{nome}: colunas faltando -> {faltando}"
    assert df["label"].isin([0, 1]).all(), f"{nome}: há label fora de {{0,1}}"
    assert df["texto"].notna().all() and (df["texto"].str.len() > 0).all(), f"{nome}: texto vazio/nulo"
    neg = int((df["label"] == 0).sum()); pos = int((df["label"] == 1).sum())
    bal = "✅" if neg == pos else f"⚠️ {neg}/{pos}"
    print(f"{nome:>9} | {len(df):>7,} | {neg:>6,} | {pos:>6,} | {bal}")

# Tamanhos de referência da Etapa 1: treino ~10,7k; cada célula de teste ~2,68k (1.340/classe).
n_treino  = len(dados["S1_train"])
n_celulas = {k: len(dados[k]) for k in ["S1_val", "S2", "S3", "S4"]}
assert n_treino > 5000, f"Treino pequeno demais ({n_treino:,}) — confira a Etapa 1."
assert len(set(n_celulas.values())) == 1, f"Células de teste com tamanhos diferentes: {n_celulas}"

print(f"\n✅ Fase 2.0 OK — treino = {n_treino:,} | "
      f"cada célula de teste = {next(iter(n_celulas.values())):,} (balanceadas 50/50).")
print("➡️  Próximo: Fase 2.1 — carregar XLM-R + head padrão do HF e encapsular em src/model.py.")


### 2.1 — Arquitetura do Classificador *(Task 2.1)*

- **INPUT:** `xlm-roberta-base` (Hugging Face).
- **AÇÕES:** `fixar_seed(seed)` **antes** de instanciar (a head nasce aleatória) · carregar `XLMRobertaForSequenceClassification(num_labels=2)` + `XLMRobertaTokenizerFast` · encapsular em `src/model.py` (`carregar_modelo`).
- **OUTPUT:** `src/model.py` (gerado e espelhado no Drive) + célula de sanity.
- **VERIFY:** forward com tensor dummy `[B, 128]` -> logits `[B, 2]`; contagem total ~ **278M** parâmetros.

> **Head (decisão D2):** padrão do HF — `RobertaClassificationHead` (dropout -> dense 768->768 -> tanh -> dropout -> linear 768->2) sobre o token `<s>`. Sempre treinável.

In [ ]:
# 2.1 (1/3) — Gera src/model.py (XLM-R + head padrão HF + freeze_layers) e espelha no Drive
from pathlib import Path
import shutil, sys

DIR_SRC = Path('/content/src'); DIR_SRC.mkdir(parents=True, exist_ok=True)
(DIR_SRC / '__init__.py').write_text('', encoding='utf-8')

MODULO_MODEL = r'''# -*- coding: utf-8 -*-
"""
src/model.py
============
Etapa 2 do projeto "Análise Arquitetural do Congelamento de Camadas em
Transformers Multilíngues" — carregamento do XLM-RoBERTa e congelamento C1-C4.

Fase 2.1: carregar_modelo(seed) -> (model, tokenizer).
Fase 2.2: freeze_layers(model, config) para C1-C4 (classifier sempre treinável).
"""
from __future__ import annotations

import random
import re
from typing import Dict, Tuple

import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    XLMRobertaForSequenceClassification,
    set_seed,
)

NOME_MODELO = "xlm-roberta-base"
NUM_LABELS = 2  # classificação binária: 0 = Negativo, 1 = Positivo

# Configurações de congelamento (Seção 6 da metodologia / PLAN-Etapa2).
# Cada config diz se congela os embeddings e QUAIS camadas do encoder (0..11).
# A classification head (classifier.*) é SEMPRE treinável (nasce do zero).
CONFIGS: Dict[str, Dict] = {
    "C1": {"nome": "Full Fine-Tuning", "embeddings": False, "camadas": set()},
    "C2": {"nome": "Freeze Lower",     "embeddings": True,  "camadas": set(range(0, 6))},
    "C3": {"nome": "Freeze Upper",     "embeddings": False, "camadas": set(range(6, 12))},
    "C4": {"nome": "Frozen Encoder",   "embeddings": True,  "camadas": set(range(0, 12))},
}

# Captura o índice i de "roberta.encoder.layer.<i>." (evita o bug de startswith
# em que "layer.1" casaria com 1, 10 e 11).
_PADRAO_CAMADA = re.compile(r"^roberta\.encoder\.layer\.(\d+)\.")


def fixar_seed(seed: int) -> None:
    """Fixa todas as fontes de aleatoriedade (idêntico à Etapa 1).

    Chamar ANTES de instanciar: a classification head do XLM-R nasce com pesos
    aleatórios, então a seed afeta a inicialização (crítico no C4, só-head).
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def carregar_modelo(
    seed: int = 42,
    nome_modelo: str = NOME_MODELO,
    num_labels: int = NUM_LABELS,
) -> Tuple[XLMRobertaForSequenceClassification, "object"]:
    """Carrega o XLM-RoBERTa-base com head de classificação binária + tokenizer.

    Head padrão do HF (RobertaClassificationHead: dropout -> dense 768->768 ->
    tanh -> dropout -> linear 768->num_labels sobre o token <s>). Tokenizer via
    AutoTokenizer(use_fast=True) para garantir o backend rápido. A seed é fixada
    ANTES da instanciação para tornar a head reprodutível.
    """
    fixar_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(nome_modelo, use_fast=True)
    model = XLMRobertaForSequenceClassification.from_pretrained(
        nome_modelo, num_labels=num_labels
    )
    return model, tokenizer


def freeze_layers(model, config: str) -> Dict[str, int]:
    """Aplica o congelamento da config (C1-C4) via requires_grad em named_parameters.

    Idempotente: define requires_grad para TODOS os parâmetros, então pode ser
    reaplicada / trocada de config sem reload do modelo. A classifier.* fica
    SEMPRE treinável. Retorna contar_parametros(model) após o congelamento.
    """
    if config not in CONFIGS:
        raise ValueError("config inválida: %r. Use uma de %s." % (config, list(CONFIGS)))
    cfg = CONFIGS[config]
    for nome, p in model.named_parameters():
        if nome.startswith("classifier"):
            p.requires_grad = True
            continue
        congelar = False
        if nome.startswith("roberta.embeddings"):
            congelar = cfg["embeddings"]
        else:
            m = _PADRAO_CAMADA.match(nome)
            if m is not None:
                congelar = int(m.group(1)) in cfg["camadas"]
        p.requires_grad = not congelar
    return contar_parametros(model)


def contar_parametros(model) -> Dict[str, int]:
    """Retorna {'total', 'treinavel', 'congelado'} em número de parâmetros."""
    total = sum(p.numel() for p in model.parameters())
    treinavel = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "treinavel": treinavel, "congelado": total - treinavel}


if __name__ == "__main__":
    # Auto-teste minimalista (CPU): carrega e exercita o congelamento C1-C4.
    model, tokenizer = carregar_modelo(seed=42)
    assert tokenizer.is_fast, "tokenizer deveria ser fast"
    head = sum(p.numel() for n, p in model.named_parameters() if n.startswith("classifier"))
    c1 = freeze_layers(model, "C1")
    assert c1["treinavel"] == c1["total"], c1
    c4 = freeze_layers(model, "C4")
    assert c4["treinavel"] == head, (c4, head)
    for cfg in CONFIGS:
        freeze_layers(model, cfg)
        assert all(p.requires_grad for n, p in model.named_parameters()
                   if n.startswith("classifier"))
    print("model OK -> freeze C1-C4 | head =", head, "| total =", c1["total"])
'''

(DIR_SRC / 'model.py').write_text(MODULO_MODEL, encoding='utf-8')
shutil.copytree('/content/src', DIR_PROJ / 'src', dirs_exist_ok=True)  # persiste no Drive
for p in ('/content', str(DIR_PROJ)):
    if p not in sys.path: sys.path.insert(0, p)
print('OK src/model.py gerado (%d bytes) e espelhado em %s' % ((DIR_SRC/'model.py').stat().st_size, DIR_PROJ/'src'))

In [ ]:
# 2.1 (2/3) — Importa o módulo e carrega o modelo (download ~1.1 GB na 1a vez)
import sys
for m in ('src.model', 'src'):
    sys.modules.pop(m, None)
from src import model as M

model_xlmr, tokenizer = M.carregar_modelo(seed=SEEDS[0])
model_xlmr.to(device)

print("Modelo   :", model_xlmr.config.name_or_path, "| num_labels =", model_xlmr.config.num_labels)
print("Tokenizer:", type(tokenizer).__name__, "| vocab =", "{:,}".format(tokenizer.vocab_size))
print("Head     :", type(model_xlmr.classifier).__name__,
      "(dense 768->768 -> tanh -> dropout -> out_proj 768->2)")

In [ ]:
# 2.1 (3/3) — VERIFY: forward dummy [B,128] -> logits [B,2]; contagem de parâmetros ~278M
import torch

B, L = 4, 128
dummy_ids  = torch.randint(0, tokenizer.vocab_size, (B, L), device=device)
dummy_mask = torch.ones_like(dummy_ids)

model_xlmr.eval()
with torch.no_grad():
    saida = model_xlmr(input_ids=dummy_ids, attention_mask=dummy_mask)

assert tuple(saida.logits.shape) == (B, 2), saida.logits.shape
cont = M.contar_parametros(model_xlmr)
emb  = model_xlmr.roberta.embeddings.word_embeddings.weight.numel()

print("logits          :", tuple(saida.logits.shape), " (esperado (%d, 2)) OK" % B)
print("parametros total: {:>13,}  (~278M esperado)".format(cont['total']))
print("  word embeddings: {:>13,}  ({:.0f}% do modelo)".format(emb, 100*emb/cont['total']))
print("  treinaveis     : {:>13,}  (tudo treinavel — congelamento entra na 2.2)".format(cont['treinavel']))
assert cont['total'] > 270_000_000, cont

print()
print("OK Fase 2.1 — XLM-R-base + head padrao HF carregado e validado.")
print("Proximo: Fase 2.2 — freeze_layers(model, config) para C1-C4.")

### 2.2 — Lógica de Congelamento C1–C4 *(Task 2.2)*

- **INPUT:** `model` + config ∈ {C1, C2, C3, C4}.
- **AÇÕES:** `freeze_layers(model, config)` seta `requires_grad=False` por prefixo em `named_parameters`. `classifier.*` **sempre** treinável.
- **OUTPUT:** `freeze_layers` em `src/model.py` (retorna a contagem) + tabela real de treináveis.
- **VERIFY:** C1 treina tudo · C4 treina só a head · C2 < C3 < C1 · head sempre treinável · prefixos corretos.

| Config | Nome | Congela | Treinável |
|---|---|---|---|
| **C1** | Full FT | — | tudo |
| **C2** | Freeze Lower (H1) | embeddings + camadas 0–5 | camadas 6–11 + head |
| **C3** | Freeze Upper (H2) | camadas 6–11 | embeddings + camadas 0–5 + head |
| **C4** | Frozen Encoder | embeddings + camadas 0–11 | só a head |

> Como os word-embeddings são ~192M (69%), C2/C4 (que os congelam) treinam bem menos do que pareceria — isso corrige os números `[CONFIRMAR]` da §6.

In [ ]:
# 2.2 (1/2) — Aplica C1–C4 e monta a TABELA REAL de parâmetros treináveis
import pandas as pd

linhas = []
for cfg in ["C1", "C2", "C3", "C4"]:
    cont = M.freeze_layers(model_xlmr, cfg)
    assert all(p.requires_grad for n, p in model_xlmr.named_parameters()
               if n.startswith("classifier")), f"{cfg}: head deveria estar treinável"
    linhas.append({
        "Config": cfg,
        "Nome": M.CONFIGS[cfg]["nome"],
        "Treináveis": cont["treinavel"],
        "Congelados": cont["congelado"],
        "% treinável": 100 * cont["treinavel"] / cont["total"],
    })

tab = pd.DataFrame(linhas)
tab_fmt = tab.assign(
    **{"Treináveis": tab["Treináveis"].map("{:,}".format),
       "Congelados": tab["Congelados"].map("{:,}".format),
       "% treinável": tab["% treinável"].map("{:.2f}%".format)}
)
print("Tabela real de parâmetros treináveis por config (fecha o [CONFIRMAR] da §6):")
display(tab_fmt)

In [ ]:
# 2.2 (2/2) — VERIFY: sanidade da lógica de congelamento
import re

total = M.contar_parametros(model_xlmr)["total"]
head  = sum(p.numel() for n, p in model_xlmr.named_parameters() if n.startswith("classifier"))
treinaveis = {cfg: M.freeze_layers(model_xlmr, cfg)["treinavel"] for cfg in ["C1", "C2", "C3", "C4"]}

assert treinaveis["C1"] == total,  f"C1 deveria treinar tudo: {treinaveis['C1']} vs {total}"
assert treinaveis["C4"] == head,   f"C4 deveria treinar só a head ({head:,}): {treinaveis['C4']:,}"
assert treinaveis["C2"] < treinaveis["C3"] < treinaveis["C1"], f"ordem C2<C3<C1 falhou: {treinaveis}"
assert tokenizer.is_fast, "tokenizer não é fast (AutoTokenizer use_fast)"

# Varredura por prefixo na C2: embeddings+0–5 congelados; 6–11 treináveis; head treinável.
M.freeze_layers(model_xlmr, "C2")
pat = re.compile(r"^roberta\.encoder\.layer\.(\d+)\.")
for n, p in model_xlmr.named_parameters():
    if n.startswith("classifier"):
        assert p.requires_grad, f"C2: head congelada? {n}"
    elif n.startswith("roberta.embeddings"):
        assert not p.requires_grad, f"C2: embeddings deveria estar congelado: {n}"
    else:
        m = pat.match(n)
        if m:
            i = int(m.group(1))
            esperado_treinavel = i >= 6
            assert p.requires_grad == esperado_treinavel, f"C2: camada {i} requires_grad errado: {n}"

M.freeze_layers(model_xlmr, "C1")  # restaura tudo treinável (estado limpo p/ a Fase 2.4)
print("OK Fase 2.2 — freeze_layers C1-C4 validado | tokenizer.is_fast =", tokenizer.is_fast)
print("head (classifier) =", f"{head:,}", "params | total =", f"{total:,}")
print("Proximo: Fase 2.3 (tests/test_model.py + atualizar §6) e 2.4 (smoke test fwd+bwd).")

### 2.3 — Testes de Verificação *(VERIFY da Task 2.2)*

- **AÇÕES:** `tests/test_model.py` — para cada config, varre `named_parameters` e assere `requires_grad` correto por prefixo; `classifier` sempre treinável; **contagem real** de treináveis por config (regressão contra os números da 2.2).
- **OUTPUT:** testes passando + tabela confirmada (fecha o `[CONFIRMAR]` da §6, já atualizada na metodologia).
- **VERIFY:** todos os asserts passam.

In [ ]:
# 2.3 (1/2) — Gera tests/test_model.py e espelha no Drive
from pathlib import Path
import shutil, sys

DIR_TESTS = Path('/content/tests'); DIR_TESTS.mkdir(parents=True, exist_ok=True)
(DIR_TESTS / '__init__.py').write_text('', encoding='utf-8')

MODULO_TESTS = r'''# -*- coding: utf-8 -*-
"""
tests/test_model.py
===================
Testes de verificação da Etapa 2 (Fase 2.3) — congelamento seletivo C1-C4.
Confere requires_grad por prefixo, head sempre treinável e a contagem real de
parâmetros treináveis por config. Roda standalone (python tests/test_model.py)
ou importado no notebook chamando rodar_todos(model) — sem recarregar o modelo.
"""
from __future__ import annotations

import re
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from src import model as M

_PAD = re.compile(r"^roberta\.encoder\.layer\.(\d+)\.")

# Especificação esperada por config: (congela_embeddings, camadas congeladas).
_ESPEC = {
    "C1": (False, set()),
    "C2": (True, set(range(0, 6))),
    "C3": (False, set(range(6, 12))),
    "C4": (True, set(range(0, 12))),
}

# Contagens reais confirmadas no run da Fase 2.2 (regressão).
_ESPERADO_TREINAVEL = {
    "C1": 278_045_186,
    "C2": 43_119_362,
    "C3": 235_517_954,
    "C4": 592_130,
}


def _head(model):
    return sum(p.numel() for n, p in model.named_parameters() if n.startswith("classifier"))


def teste_c1_treina_tudo(model):
    c = M.freeze_layers(model, "C1")
    assert c["treinavel"] == c["total"], c


def teste_c4_so_head(model):
    c = M.freeze_layers(model, "C4")
    assert c["treinavel"] == _head(model), (c, _head(model))


def teste_head_sempre_treinavel(model):
    for cfg in M.CONFIGS:
        M.freeze_layers(model, cfg)
        congeladas = [n for n, p in model.named_parameters()
                      if n.startswith("classifier") and not p.requires_grad]
        assert not congeladas, (cfg, congeladas)


def teste_prefixos_por_config(model):
    for cfg, (emb_cong, cam_cong) in _ESPEC.items():
        M.freeze_layers(model, cfg)
        for n, p in model.named_parameters():
            if n.startswith("classifier"):
                assert p.requires_grad, (cfg, n)
            elif n.startswith("roberta.embeddings"):
                assert p.requires_grad == (not emb_cong), (cfg, n)
            else:
                m = _PAD.match(n)
                if m is not None:
                    i = int(m.group(1))
                    assert p.requires_grad == (i not in cam_cong), (cfg, n)


def teste_contagem_real(model):
    for cfg, val in _ESPERADO_TREINAVEL.items():
        c = M.freeze_layers(model, cfg)
        assert c["treinavel"] == val, (cfg, c["treinavel"], val)


def teste_idempotente(model):
    a = M.freeze_layers(model, "C2")["treinavel"]
    b = M.freeze_layers(model, "C2")["treinavel"]
    M.freeze_layers(model, "C3")
    c = M.freeze_layers(model, "C2")["treinavel"]
    assert a == b == c, (a, b, c)


def teste_config_invalida(model):
    try:
        M.freeze_layers(model, "C9")
    except ValueError:
        return
    raise AssertionError("freeze_layers deveria levantar ValueError para config inválida")


TESTES = [
    teste_c1_treina_tudo,
    teste_c4_so_head,
    teste_head_sempre_treinavel,
    teste_prefixos_por_config,
    teste_contagem_real,
    teste_idempotente,
    teste_config_invalida,
]


def rodar_todos(model):
    falhas = []
    for t in TESTES:
        try:
            t(model)
            print("  PASS", t.__name__)
        except AssertionError as e:
            falhas.append(t.__name__)
            print("  FAIL", t.__name__, "->", e)
    M.freeze_layers(model, "C1")  # restaura estado limpo
    if falhas:
        raise AssertionError("%d teste(s) falharam: %s" % (len(falhas), falhas))
    print("OK %d testes passaram." % len(TESTES))
    return True


if __name__ == "__main__":
    modelo, _ = M.carregar_modelo(seed=42)
    rodar_todos(modelo)
'''

(DIR_TESTS / 'test_model.py').write_text(MODULO_TESTS, encoding='utf-8')
shutil.copytree('/content/tests', DIR_PROJ / 'tests', dirs_exist_ok=True)  # persiste no Drive
if '/content' not in sys.path: sys.path.insert(0, '/content')
print('OK tests/test_model.py gerado (%d bytes) e espelhado em %s' % ((DIR_TESTS/'test_model.py').stat().st_size, DIR_PROJ/'tests'))

In [ ]:
# 2.3 (2/2) — Executa os testes formais contra o modelo carregado (sem reload)
import sys
for m in ('tests.test_model', 'tests'):
    sys.modules.pop(m, None)
from tests import test_model as T

print("Rodando", len(T.TESTES), "testes de congelamento:")
T.rodar_todos(model_xlmr)
print("OK Fase 2.3 — testes formais passaram; §6 da metodologia atualizada com os números reais.")

### 2.4 — Smoke Test com Dados Reais (forward + backward)

- **AÇÕES:** tokenizar 1 batch real de `S1_train` (max_len=128) → forward → loss (Cross-Entropy) → backward, para **cada** config.
- **VERIFY:** `.grad` **nulo nos parâmetros congelados** e **não-nulo nos treináveis** (em especial a head); loss finita. É a prova de que o congelamento funciona no fluxo real, não só no `requires_grad`.

In [ ]:
# 2.4 — Smoke test: 1 batch real de S1_train; forward+backward por config.
#       Prova que o gradiente flui SO nos parametros treinaveis.
import torch

B = 8
amostra = dados["S1_train"].sample(B, random_state=SEEDS[0])
enc = tokenizer(list(amostra["texto"]), truncation=True, padding="max_length",
                max_length=128, return_tensors="pt").to(device)
labels = torch.tensor(amostra["label"].tolist(), device=device)

def smoke(model, config):
    M.freeze_layers(model, config)
    model.train(); model.zero_grad(set_to_none=True)
    out = model(**enc, labels=labels)
    out.loss.backward()
    congelados_com_grad, treina_com_grad, head_ok = [], 0, True
    for n, p in model.named_parameters():
        grad_nz = (p.grad is not None) and bool(torch.any(p.grad != 0))
        if p.requires_grad:
            treina_com_grad += int(grad_nz)
            if n.startswith("classifier") and not grad_nz:
                head_ok = False
        elif grad_nz:
            congelados_com_grad.append(n)
    return out.loss.item(), treina_com_grad, congelados_com_grad, head_ok

print(f"{'config':>6} | {'loss':>8} | {'tensores treinaveis c/ grad':>27} | {'congelados c/ grad':>18}")
print('-' * 70)
for cfg in ["C1", "C2", "C3", "C4"]:
    loss, treina, congel, head_ok = smoke(model_xlmr, cfg)
    assert torch.isfinite(torch.tensor(loss)), f"{cfg}: loss nao finita ({loss})"
    assert not congel, f"{cfg}: {len(congel)} tensores CONGELADOS receberam gradiente! ex: {congel[:2]}"
    assert head_ok, f"{cfg}: a head (classifier) NAO recebeu gradiente"
    print(f"{cfg:>6} | {loss:8.4f} | {treina:>27,} | {len(congel):>18}")

M.freeze_layers(model_xlmr, "C1"); model_xlmr.zero_grad(set_to_none=True)
print("\nOK Fase 2.4 — gradiente flui SO nos treinaveis (loss finita; 0 congelados com grad; head sempre com grad).")
print("ETAPA 2 COMPLETA: src/model.py (carregar_modelo + freeze_layers) testado e pronto p/ a Etapa 3.")

### ✅ Etapa 2 — Checklist final

| Critério (do PLAN-Etapa2) | Status |
|---|---|
| Modelo carrega, logits `[batch, 2]`, ~278M params | ✔️ Fase 2.1 |
| `freeze_layers` cobre C1–C4; `classifier` sempre treinável | ✔️ Fase 2.2 |
| Tabela real de parâmetros treináveis por config (fecha §6) | ✔️ Fases 2.2–2.3 |
| `tests/test_model.py` passando | ✔️ Fase 2.3 |
| Smoke test: gradiente só pelos parâmetros não congelados | ✔️ Fase 2.4 |
| `src/model.py` versionado e importável pela Etapa 3 | ✔️ |

**Entregáveis no Drive (`MyDrive/TrabalhoRNP/`):** `src/model.py`, `tests/test_model.py`, os 5 CSVs.

> **Próximo: Etapa 3** — pipeline de treino multi-seed (12 treinos = 4 configs × 3 seeds `{42,123,2024}`): AdamW lr 2e-5, warmup 10%, `fp16`, early stopping (paciência 1, `eval_loss`), 3 épocas. Salva curvas de loss + checkpoints no Drive.